In [1]:
# Projeto KDD: Avaliação Formativa por Módulos
# Agrupamento de tópicos matemáticos via K-Means e priorização local.

import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from google.colab import drive

# Configurações iniciais
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 280)

CAMINHO_ARQUIVO = "/content/drive/MyDrive/2012-2013-data-with-predictions-4-final.csv"
PASTA_SAIDA = Path("/content/drive/MyDrive/resultados_kdd_modulos_sem_afeto")

COLUNAS_SELECIONADAS = [
    "skill",
    "correct",
    "attempt_count",
    "hint_count",
    "ms_first_response"
]

COLUNAS_MODELO = [
    "taxa_erro",
    "attempt_count",
    "hint_count",
    "sec_first_response"
]

PESOS_SCORE = {
    "taxa_erro_z": 0.50,
    "sec_first_response_z": 0.25,
    "attempt_count_z": 0.15,
    "hint_count_z": 0.10
}

MIN_REGISTROS_SKILL = 50
K_GLOBAL = 3
TOP_N = 3

MODULO_ATUAL = "Módulo 2: Frações"

ANALISAR_TODOS_MODULOS = True
RODAR_DIAGNOSTICO_CLUSTER_LOCAL = True

# Módulos pedagógicos manuais
modulos_pedagogicos = {
    "Módulo 1: Números, Operações e Propriedades": [
        "Addition Whole Numbers", "Subtraction Whole Numbers", "Multiplication Whole Numbers",
        "Division Whole Numbers", "Odd and Even Number", "Prime Number", "Prime Factor",
        "Divisibility Rules", "Common Factor", "Common Multiple", "Greatest Common Factor",
        "Least Common Multiple", "Order of Operations +,-,/,* () positive reals",
        "Order of Operations All", "Computation with Real Numbers", "Absolute Value",
        "Square Root", "Square Roots", "Exponents", "Scientific Notation", "Rounding",
        "Estimation", "Expanded, Standard and Word Notation", "Associative Property",
        "Commutative Property", "Distributive Property"
    ],
    "Módulo 2: Frações": [
        "Equivalent Fractions", "Ordering Fractions", "Fraction Of", "Addition Proper Fractions",
        "Subtraction Proper Fractions", "Multiplication Proper Fractions", "Division Proper Fractions",
        "Addition Mixed Fractions", "Subtraction Mixed Fractions", "Multiplication Mixed Fractions",
        "Division Mixed Fractions", "Addition and Subtraction Fractions", "Multiplication Fractions",
        "Division Fractions"
    ],
    "Módulo 3: Decimais, Porcentagem, Razão e Proporção": [
        "Addition and Subtraction Positive Decimals", "Multiplication Positive Decimals",
        "Multiplication and Division Positive Decimals", "Ordering Positive Decimals",
        "Conversion of Fraction Decimals Percents", "Percents", "Percent Of", "Finding Percents",
        "Percent Discount", "Percent Increase or Decrease", "Finding Ratios", "Rate", "Unit Rate",
        "Proportion", "Scale Factor", "Effect of Changing Dimensions of a Shape Prportionally"
    ],
    "Módulo 4: Inteiros e Reta Numérica": [
        "Number Line", "Ordering Integers", "Addition and Subtraction Integers",
        "Multiplication and Division Integers", "Ordering Real Numbers", "Ordering Whole Numbers",
        "Graphing Inequalities on a number line"
    ],
    "Módulo 5: Álgebra - Expressões e Polinômios": [
        "Algebraic Simplification", "Algebraic Solving", "Combining Like Terms",
        "Simplifying Expressions positive exponents", "Picking Expressions From Choices",
        "Recognizing Equivalent Expressions", "Parts of a Polyomial, Terms, Coefficient, Monomial, Exponent, Variable",
        "Polynomial Factors", "Factoring Polynomials Standard", "Factoring Trinomials",
        "Multiplying non Monomial Polynomials", "Writine Expression from Diagrams"
    ],
    "Módulo 6: Equações, Inequações e Sistemas": [
        "Equation Solving Two or Fewer Steps", "Equation Solving More Than Two Steps",
        "Choose an Equation from Given Information", "Picking Equation and Inequality from Choices",
        "Solving for a variable", "Solving Inequalities", "Solving System of Equation",
        "Solving Systems of Linear Equations", "Solving Systems of Linear Equations by Graphing",
        "Substitution", "Quadratic Equation Solving", "Quadratic Formula to Solve Quadratic Equation",
        "Solve Quadratic Equations Using Factoring"
    ],
    "Módulo 7: Funções Lineares, Gráficos e Coordenadas": [
        "Graphing Linear Equations", "Write Linear Equation from Graph", "Write Linear Equation from Ordered Pairs",
        "Write Linear Equation from Situation", "Write Linear Equation from Slope and y-intercept",
        "Finding Slope From Equation", "Finding Slope From Situation", "Finding Slope from Graph",
        "Finding Slope from Ordered Pairs", "Finding y-intercept from Linear Equation",
        "Comparing and Identifying Slope/Rate of Change", "Making a Table from an Equation",
        "Interpreting Coordinate Graphs", "X-Y Graph Reading", "Point Plotting", "Line of Best-Fit",
        "Recognize Linear Pattern", "Composition of Function Adding", "Inverse Relations", "Pattern Finding"
    ],
    "Módulo 8: Geometria Plana - Formas, Ângulos e Perímetro": [
        "Geometric Definitions", "Angles - Obtuse, Acute, and Right", "Angles on Parallel Lines Cut by a Transversal",
        "Complementary and Supplementary Angles", "Interior Angles Triangle", "Interior Angles Figures with More than 3 Sides",
        "Properties and Classification Triangles", "Properties and Classification Quadrilaterals",
        "Properties and Classification Polygons with 5 or more sides", "Properties and Classification Circle",
        "Circle Concept", "Definition Pi", "Circumference", "Perimeter of a Polygon", "Parallel and Perpendicular Lines",
        "Congruence", "Line Symmetry", "Reflection", "Rotations", "Translations", "Transformation",
        "Midpoint", "Distance Formula", "Pythagorean Theorem"
    ],
    "Módulo 9: Áreas": [
        "Area Rectangle", "Area Triangle", "Area Circle", "Area Parallelogram", "Area Trapezoid", "Area Irregular Figure"
    ],
    "Módulo 10: Geometria Espacial - Volume e Superfície": [
        "Nets of 3D Figures", "Nets of 3D Objects", "Properties and Classification Prism",
        "Properties and Classification Rectangular Prisms", "Properties and Clasification of Pyramid",
        "Concept Volume", "Volume Cone", "Volume Cylinder", "Volume Prism", "Volume Pyramid",
        "Volume Rectangular Prism", "Volume Sphere", "Volume of 3D Objects", "Surface Area Cylinder",
        "Surface Area Pyramid", "Surface Area Rectangular Prism", "Surface Area Sphere",
        "Surface Area of 3D Objects", "Surface Area of Prism", "Calculations with Similar Figures"
    ],
    "Módulo 11: Medidas, Unidades e Tempo": [
        "Reading a Ruler or Scale", "Elapsed Time", "English and Metric Terminology",
        "Unit Conversion Standard to Metric", "Unit Conversion Within a System"
    ],
    "Módulo 12: Estatística e Representação de Dados": [
        "Mean", "Median", "Mode", "Range", "Mean-Median-Mode-Range Differentiation", "Bar Graph",
        "Circle Graph", "Histogram as Table or Graph", "Line Plot", "Scatter Plot", "Stem and Leaf Plot",
        "Box and Whisker", "Table", "Graph Shape", "Sampling Techniques"
    ],
    "Módulo 13: Probabilidade e Combinatória": [
        "Probability of a Single Event", "Probability of Two Distinct Events",
        "D.4.8-understanding-concept-of-probabilities", "Counting Methods",
        "Tree Diagrams, Lists for Counting", "Venn Diagram"
    ]
}

# Funções auxiliares de processamento
def salvar_csv(df, nome_arquivo, index=False):
    caminho = PASTA_SAIDA / nome_arquivo
    df.to_csv(caminho, index=index, encoding="utf-8-sig")
    print(f"-> Arquivo salvo: {caminho}")

def verificar_colunas_obrigatorias(caminho, colunas_obrigatorias):
    df_header = pd.read_csv(caminho, nrows=0)
    colunas_disponiveis = df_header.columns.tolist()
    ausentes = [col for col in colunas_obrigatorias if col not in colunas_disponiveis]
    if ausentes:
        raise ValueError(f"Colunas obrigatórias ausentes no arquivo: {ausentes}")
    print("-> Todas as colunas obrigatórias foram encontradas.")

def verificar_intersecoes_modulos(modulos):
    ocorrencias = defaultdict(list)
    for nome_modulo, lista_skills in modulos.items():
        for skill in lista_skills:
            ocorrencias[skill].append(nome_modulo)

    repetidas = {skill: lista_modulos for skill, lista_modulos in ocorrencias.items() if len(lista_modulos) > 1}

    print("\n" + "=" * 100)
    print("VERIFICAÇÃO DE INTERSEÇÕES ENTRE MÓDULOS")
    print("=" * 100)

    if not repetidas:
        print("Nenhuma skill aparece em mais de um módulo.")
    else:
        print(f"Foram encontradas {len(repetidas)} skill(s) em mais de um módulo:\n")
        for skill, lista_modulos in repetidas.items():
            print(f"- {skill}")
            for modulo in lista_modulos:
                print(f"  -> {modulo}")
    return repetidas

def gerar_mapeamento_modulos(modulos):
    linhas = [{"modulo": nome_modulo, "skill": skill} for nome_modulo, lista_skills in modulos.items() for skill in lista_skills]
    return pd.DataFrame(linhas)

def gerar_resumo_modulos(df_global, modulos):
    linhas = []
    for nome_modulo, lista_skills in modulos.items():
        skills_definidas = list(dict.fromkeys(lista_skills))
        skills_encontradas = [skill for skill in skills_definidas if skill in df_global.index]
        skills_ausentes = [skill for skill in skills_definidas if skill not in df_global.index]

        registros = int(df_global.loc[skills_encontradas, "quantidade"].sum()) if skills_encontradas else 0

        linhas.append({
            "modulo": nome_modulo,
            "skills_definidas_manual": len(skills_definidas),
            "skills_encontradas_apos_filtro": len(skills_encontradas),
            "skills_ausentes_apos_filtro": len(skills_ausentes),
            "registros_considerados": registros
        })

    df_resumo = pd.DataFrame(linhas)
    print("\n" + "=" * 100)
    print("RESUMO DE COBERTURA DOS MÓDULOS")
    print("=" * 100)
    print(df_resumo.to_string(index=False))
    return df_resumo

def avaliar_clusterizacao_local_por_modulo(df_global, modulos, colunas_modelo, k_max=5):
    # Diagnóstico complementar para validar agrupamentos locais
    linhas = []
    for nome_modulo, lista_skills in modulos.items():
        skills_encontradas = [skill for skill in lista_skills if skill in df_global.index]
        df_modulo = df_global.loc[skills_encontradas].copy()
        n_topicos = len(df_modulo)

        if n_topicos < 4:
            linhas.append({
                "modulo": nome_modulo,
                "topicos": n_topicos,
                "melhor_k_local": None,
                "melhor_silhueta_local": None,
                "observacao": "Poucos tópicos para clusterização local estável"
            })
            continue

        X_local = df_modulo[colunas_modelo]
        X_local_escalado = StandardScaler().fit_transform(X_local)

        limite_k = min(k_max, n_topicos - 1)
        melhor_k = None
        melhor_score = -1

        for k in range(2, limite_k + 1):
            try:
                km = KMeans(n_clusters=k, random_state=42, n_init=10)
                labels = km.fit_predict(X_local_escalado)
                score = silhouette_score(X_local_escalado, labels)

                if score > melhor_score:
                    melhor_score = score
                    melhor_k = k
            except Exception:
                pass

        linhas.append({
            "modulo": nome_modulo,
            "topicos": n_topicos,
            "melhor_k_local": melhor_k,
            "melhor_silhueta_local": round(melhor_score, 3) if melhor_k is not None else None,
            "observacao": "Diagnóstico local; método principal permanece global"
        })

    df_diagnostico = pd.DataFrame(linhas)
    print("\n" + "=" * 100)
    print("DIAGNÓSTICO COMPLEMENTAR: CLUSTERIZAÇÃO LOCAL POR MÓDULO")
    print("=" * 100)
    print(df_diagnostico.to_string(index=False))
    return df_diagnostico

def classificar_forca_indicio(qtd_sinais):
    if qtd_sinais >= 3: return "Alta"
    if qtd_sinais == 2: return "Média"
    if qtd_sinais == 1: return "Baixa"
    return "Sem indício"

def adicionar_hipoteses_comportamentais(df_global):
    # Levanta hipóteses sobre padrões de erro usando apenas variáveis comportamentais
    df = df_global.copy()
    hipoteses, justificativas, forcas, qtd_sinais_lista = [], [], [], []

    for _, linha in df.iterrows():
        perfil = linha["perfil_global"]
        erro_acima_media = linha["taxa_erro_z"] > 0
        erro_muito_alto = linha["taxa_erro_z"] > 1
        tempo_alto = linha["sec_first_response_z"] > 0.5
        tempo_baixo = linha["sec_first_response_z"] < -0.5
        tentativas_altas = linha["attempt_count_z"] > 0.5
        tentativas_baixas = linha["attempt_count_z"] < -0.5
        dicas_altas = linha["hint_count_z"] > 0.5
        dicas_baixas = linha["hint_count_z"] < -0.5

        apoio_alto = tentativas_altas or dicas_altas
        apoio_baixo = tentativas_baixas and dicas_baixas

        if perfil == "Consolidado":
            if erro_muito_alto and tempo_baixo and apoio_baixo:
                hipotese = "Atenção pontual: erro alto com resposta rápida"
                justificativa = "Apesar de estar no perfil consolidado, o tópico apresenta taxa de erro elevada, tempo baixo, poucas tentativas e baixo uso de dicas. Isso pode indicar erro rápido ou baixa persistência em parte dos registros."
                sinais = 3
                forca = "Média"
            elif erro_muito_alto:
                hipotese = "Atenção pontual: erro acima do esperado"
                justificativa = "Apesar de estar no perfil consolidado, o tópico apresenta taxa de erro acima do esperado. Como os demais indicadores não sustentam um perfil crítico, o caso deve ser tratado como atenção pontual, não como dificuldade ampla."
                sinais = 2
                forca = "Baixa"
            else:
                hipotese = "Sem indício forte de dificuldade"
                justificativa = "O tópico está no perfil consolidado e não apresenta combinação forte de erro, tempo, tentativas e dicas."
                sinais = 0
                forca = "Sem indício"
        else:
            sinais = 0
            if erro_acima_media: sinais += 1
            if erro_muito_alto: sinais += 1
            if tempo_alto or tempo_baixo: sinais += 1
            if tentativas_altas or tentativas_baixas: sinais += 1
            if dicas_altas or dicas_baixas: sinais += 1

            if erro_acima_media and tempo_alto and apoio_alto:
                hipotese = "Possível dificuldade conceitual com esforço"
                justificativa = "O tópico apresenta erro acima da média, tempo de resposta alto e sinais de esforço por tentativas e/ou uso de dicas."
            elif erro_acima_media and tempo_alto and apoio_baixo:
                hipotese = "Possível dificuldade sem uso de apoio"
                justificativa = "O tópico apresenta erro acima da média e tempo alto, mas com poucas tentativas e baixo uso de dicas."
            elif erro_acima_media and tempo_baixo and apoio_baixo:
                hipotese = "Possível baixa persistência ou resposta rápida"
                justificativa = "O tópico apresenta erro acima da média, tempo baixo, poucas tentativas e baixo uso de dicas."
            elif erro_acima_media and dicas_altas:
                hipotese = "Possível dificuldade com busca de ajuda"
                justificativa = "O tópico apresenta erro acima da média junto com maior uso de dicas."
            elif erro_acima_media and tentativas_altas:
                hipotese = "Possível dificuldade com múltiplas tentativas"
                justificativa = "O tópico apresenta erro acima da média junto com maior número de tentativas."
            elif erro_acima_media:
                hipotese = "Erro alto sem padrão comportamental claro"
                justificativa = "O tópico apresenta erro acima da média, mas os demais indicadores não formam um padrão comportamental claro."
            else:
                hipotese = "Sem indício forte de dificuldade"
                justificativa = "O tópico não apresenta combinação forte de erro, tempo, tentativas e dicas."
                sinais = 0

            if perfil == "Crítico" and sinais >= 2:
                forca = "Alta"
            elif sinais >= 3:
                forca = "Alta"
            elif sinais == 2:
                forca = "Média"
            elif sinais == 1:
                forca = "Baixa"
            else:
                forca = "Sem indício"

        hipoteses.append(hipotese)
        justificativas.append(justificativa)
        qtd_sinais_lista.append(sinais)
        forcas.append(forca)

    df["hipotese_padrao_erro"] = hipoteses
    df["justificativa_hipotese"] = justificativas
    df["qtd_sinais_hipotese"] = qtd_sinais_lista
    df["forca_indicio_hipotese"] = forcas

    df["observacao_hipotese"] = "Hipótese interpretável baseada apenas em indicadores comportamentais; não representa causa real comprovada."

    print("\n" + "=" * 100)
    print("RESUMO DAS HIPÓTESES COMPORTAMENTAIS SOBRE PADRÕES DE ERRO")
    print("=" * 100)
    print(df["hipotese_padrao_erro"].value_counts(dropna=False).to_string())
    return df

def validar_modulo(df_treinado, nome_modulo, lista_skills, top_n=3):
    print("\n" + "=" * 130)
    print(f"RELATÓRIO DE VALIDAÇÃO FORMATIVA — {nome_modulo.upper()}")
    print("=" * 130)

    skills_encontradas = [skill for skill in lista_skills if skill in df_treinado.index]
    skills_ausentes = [skill for skill in lista_skills if skill not in df_treinado.index]

    print(f"\nSkills definidas manualmente no módulo: {len(lista_skills)}")
    print(f"Skills encontradas após limpeza/filtro: {len(skills_encontradas)}")
    print(f"Skills ausentes após limpeza/filtro: {len(skills_ausentes)}")

    if skills_ausentes:
        print(f"Ausentes: {skills_ausentes}")

    if not skills_encontradas:
        print("Nenhum tópico deste módulo foi encontrado na base após os filtros.")
        return None

    df_modulo = df_treinado.loc[skills_encontradas].copy()
    df_modulo = df_modulo.sort_values(by="score_prioridade_local", ascending=False).copy()
    df_modulo["rank_local"] = range(1, len(df_modulo) + 1)

    cond_prioridade_local = (
        (df_modulo["rank_local"] <= top_n) &
        (df_modulo["score_prioridade_local"] > 0) &
        (df_modulo["perfil_global"] != "Consolidado")
    )

    df_modulo["prioridade_local"] = "-"
    df_modulo.loc[cond_prioridade_local, "prioridade_local"] = f"Top {top_n} local"

    df_modulo["recomendado"] = (
        (df_modulo["perfil_global"] == "Crítico") |
        (df_modulo["prioridade_local"] != "-")
    )

    def definir_motivo(linha):
        critico = linha["perfil_global"] == "Crítico"
        prioridade = linha["prioridade_local"] != "-"
        if critico and prioridade: return "Crítico global + prioridade local"
        if critico: return "Crítico global"
        if prioridade: return "Prioridade local"
        return "-"

    df_modulo["motivo_recomendacao"] = df_modulo.apply(definir_motivo, axis=1)

    baseline_erro = df_modulo.sort_values(by="taxa_erro", ascending=False).head(top_n)
    df_recomendados = df_modulo[df_modulo["recomendado"]].copy()

    colunas_relatorio = [
        "perfil_global", "taxa_erro", "sec_first_response", "attempt_count", "hint_count",
        "score_prioridade_local", "rank_local", "hipotese_padrao_erro", "forca_indicio_hipotese",
        "justificativa_hipotese", "prioridade_local", "motivo_recomendacao"
    ]

    print(f"\nResumo: {len(df_modulo)} subtópico(s) encontrados no módulo.")
    print("\nVISÃO INTEGRADA DO MÓDULO:")
    print(df_modulo[colunas_relatorio].round(3).reset_index().rename(columns={"skill": "topico"}).to_string(index=False))

    print("\n" + "-" * 130)
    print("TÓPICOS RECOMENDADOS PELO MÉTODO PROPOSTO")
    print("(Crítico global OU prioridade local válida)")
    print("-" * 130)

    if df_recomendados.empty:
        print("Nenhum tópico foi recomendado para reforço segundo os critérios definidos.")
    else:
        print(df_recomendados[["perfil_global", "taxa_erro", "score_prioridade_local", "hipotese_padrao_erro", "forca_indicio_hipotese", "motivo_recomendacao"]].round(3).reset_index().rename(columns={"skill": "topico"}).to_string(index=False))

    print("\n" + "-" * 130)
    print("COMPARAÇÃO COM BASELINE SIMPLES")
    print("(Baseline = escolher apenas os Top N tópicos com maior taxa de erro)")
    print("-" * 130)

    print("\nBaseline por taxa de erro:")
    print(baseline_erro[["taxa_erro", "perfil_global", "score_prioridade_local", "hipotese_padrao_erro"]].round(3).reset_index().rename(columns={"skill": "topico"}).to_string(index=False))

    print("\nMétodo proposto:")
    if df_recomendados.empty:
        print("Nenhum tópico recomendado.")
    else:
        print(df_recomendados[["taxa_erro", "perfil_global", "score_prioridade_local", "hipotese_padrao_erro", "motivo_recomendacao"]].round(3).reset_index().rename(columns={"skill": "topico"}).to_string(index=False))

    baseline_set = set(baseline_erro.index)
    proposto_set = set(df_recomendados.index)

    em_comum = baseline_set & proposto_set
    so_baseline = baseline_set - proposto_set
    so_proposto = proposto_set - baseline_set

    print("\nResumo comparativo:")
    print(f"- Tópicos em comum: {len(em_comum)}")
    print(f"- Apenas no baseline: {len(so_baseline)}")
    print(f"- Apenas no método proposto: {len(so_proposto)}")

    if so_baseline: print(f"- Apenas no baseline: {list(so_baseline)}")
    if so_proposto: print(f"- Apenas no método proposto: {list(so_proposto)}")

    print("\n" + "-" * 130)
    print("AÇÃO SUGERIDA:")

    qtd_criticos = (df_modulo["perfil_global"] == "Crítico").sum()
    qtd_prioridades_locais = (df_modulo["prioridade_local"] != "-").sum()

    if qtd_criticos > 0:
        print("O módulo possui tópico(s) com perfil crítico global. Recomenda-se considerar reforço pedagógico, priorizando os tópicos listados pelo método proposto.")
    elif qtd_prioridades_locais > 0:
        print("O módulo não possui tópicos críticos globalmente, mas apresenta prioridades locais válidas. O professor pode avançar no cronograma, mantendo acompanhamento pontual dos tópicos destacados.")
    else:
        print("Não foram encontrados indícios fortes de necessidade de reforço amplo neste módulo. O professor pode considerar avançar no cronograma, conforme sua avaliação pedagógica.")

    print("=" * 130)

    return {
        "modulo": nome_modulo, "df_modulo": df_modulo, "df_recomendados": df_recomendados,
        "baseline_erro": baseline_erro, "em_comum": em_comum, "so_baseline": so_baseline,
        "so_proposto": so_proposto, "skills_definidas": len(lista_skills),
        "skills_encontradas": len(skills_encontradas), "skills_ausentes": len(skills_ausentes)
    }

def gerar_resumo_comparativo(resultados):
    linhas = []
    for nome_modulo, resultado in resultados.items():
        if resultado is None:
            linhas.append({"modulo": nome_modulo, "skills_definidas": None, "skills_encontradas": 0, "skills_ausentes": None, "topicos_recomendados": 0, "topicos_baseline": 0, "em_comum": 0, "apenas_baseline": 0, "apenas_metodo": 0})
            continue

        linhas.append({
            "modulo": nome_modulo,
            "skills_definidas": resultado["skills_definidas"],
            "skills_encontradas": resultado["skills_encontradas"],
            "skills_ausentes": resultado["skills_ausentes"],
            "topicos_recomendados": len(resultado["df_recomendados"]),
            "topicos_baseline": len(resultado["baseline_erro"]),
            "em_comum": len(resultado["em_comum"]),
            "apenas_baseline": len(resultado["so_baseline"]),
            "apenas_metodo": len(resultado["so_proposto"])
        })

    df_resumo = pd.DataFrame(linhas)
    print("\n" + "=" * 100)
    print("RESUMO COMPARATIVO ENTRE BASELINE E MÉTODO PROPOSTO")
    print("=" * 100)
    print(df_resumo.to_string(index=False))
    return df_resumo

def gerar_csvs_por_modulo(resultados):
    linhas_modulos, linhas_recomendados, linhas_baseline = [], [], []

    for nome_modulo, resultado in resultados.items():
        if resultado is None: continue

        df_modulo = resultado["df_modulo"].copy().reset_index().rename(columns={"skill": "topico"})
        df_modulo.insert(0, "modulo", nome_modulo)
        linhas_modulos.append(df_modulo)

        df_rec = resultado["df_recomendados"].copy().reset_index().rename(columns={"skill": "topico"})
        df_rec.insert(0, "modulo", nome_modulo)
        linhas_recomendados.append(df_rec)

        df_base = resultado["baseline_erro"].copy().reset_index().rename(columns={"skill": "topico"})
        df_base.insert(0, "modulo", nome_modulo)
        linhas_baseline.append(df_base)

    if linhas_modulos: salvar_csv(pd.concat(linhas_modulos, ignore_index=True), "10_relatorio_detalhado_todos_modulos.csv", index=False)
    if linhas_recomendados: salvar_csv(pd.concat(linhas_recomendados, ignore_index=True), "11_topicos_recomendados_metodo_proposto.csv", index=False)
    if linhas_baseline: salvar_csv(pd.concat(linhas_baseline, ignore_index=True), "12_topicos_baseline_taxa_erro.csv", index=False)

# Execução principal do Script
print("1. Conectando ao Google Drive...")
drive.mount("/content/drive", force_remount=True)
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
print(f"-> Pasta de saída: {PASTA_SAIDA}")

print("\n2. Verificando colunas obrigatórias...")
verificar_colunas_obrigatorias(CAMINHO_ARQUIVO, COLUNAS_SELECIONADAS)

print("\n3. Carregando base de dados geral...")
df_bruto = pd.read_csv(CAMINHO_ARQUIVO, usecols=COLUNAS_SELECIONADAS)
print(f"-> Linhas brutas carregadas: {len(df_bruto):,}")

print("\n4. Executando pré-processamento...")
resumo_limpeza = []
def registrar_limpeza(etapa, df):
    resumo_limpeza.append({"etapa": etapa, "linhas": len(df)})

registrar_limpeza("Base carregada", df_bruto)

df_bruto = df_bruto.dropna(subset=["skill"]).copy()
df_bruto["skill"] = df_bruto["skill"].astype(str).str.strip()
df_bruto = df_bruto[df_bruto["skill"] != ""].copy()
registrar_limpeza("Remoção de skill ausente ou vazia", df_bruto)

for coluna in ["correct", "attempt_count", "hint_count", "ms_first_response"]:
    df_bruto[coluna] = pd.to_numeric(df_bruto[coluna], errors="coerce")

df_bruto = df_bruto.dropna(subset=["correct", "attempt_count", "hint_count", "ms_first_response"]).copy()
registrar_limpeza("Remoção de valores ausentes nas variáveis principais", df_bruto)

df_bruto["correct_binario"] = (df_bruto["correct"] == 1.0).astype(int)
df_bruto["sec_first_response"] = df_bruto["ms_first_response"] / 1000

df_bruto = df_bruto[(df_bruto["sec_first_response"] > 0) & (df_bruto["attempt_count"] >= 0) & (df_bruto["hint_count"] >= 0)].copy()
registrar_limpeza("Remoção de tempo/tentativas/dicas inválidos", df_bruto)

df_resumo_limpeza = pd.DataFrame(resumo_limpeza)
df_resumo_limpeza["removidas_desde_etapa_anterior"] = df_resumo_limpeza["linhas"].shift(1) - df_resumo_limpeza["linhas"]
df_resumo_limpeza.loc[0, "removidas_desde_etapa_anterior"] = 0
df_resumo_limpeza["percentual_da_base_original"] = (df_resumo_limpeza["linhas"] / df_resumo_limpeza.loc[0, "linhas"] * 100).round(2)

print("\nRESUMO DA LIMPEZA")
print(df_resumo_limpeza.to_string(index=False))
salvar_csv(df_resumo_limpeza, "01_resumo_limpeza.csv", index=False)
print(f"\n-> Linhas após limpeza: {len(df_bruto):,}")

print("\n5. Agregando registros por tópico matemático...")
df_global_sem_filtro = df_bruto.groupby("skill").agg(
    media_correct=("correct_binario", "mean"), attempt_count=("attempt_count", "mean"),
    hint_count=("hint_count", "mean"), sec_first_response=("sec_first_response", "median"),
    quantidade=("skill", "count")
)
df_global_sem_filtro["taxa_erro"] = 1 - df_global_sem_filtro["media_correct"]

print(f"-> Tópicos antes do filtro mínimo: {len(df_global_sem_filtro)}")
df_global = df_global_sem_filtro[df_global_sem_filtro["quantidade"] >= MIN_REGISTROS_SKILL].copy()
print(f"-> Tópicos após filtro mínimo de {MIN_REGISTROS_SKILL} registros: {len(df_global)}")

df_topicos_removidos = df_global_sem_filtro[df_global_sem_filtro["quantidade"] < MIN_REGISTROS_SKILL].copy()
if not df_topicos_removidos.empty:
    salvar_csv(df_topicos_removidos.sort_values("quantidade").reset_index(), "02_topicos_removidos_por_baixa_quantidade.csv", index=False)

df_mapeamento_modulos = gerar_mapeamento_modulos(modulos_pedagogicos)
salvar_csv(df_mapeamento_modulos, "00_mapeamento_manual_modulos_skills.csv", index=False)

intersecoes = verificar_intersecoes_modulos(modulos_pedagogicos)
df_resumo_modulos = gerar_resumo_modulos(df_global, modulos_pedagogicos)
salvar_csv(df_resumo_modulos, "03_resumo_cobertura_modulos.csv", index=False)

print("\n6. Normalizando variáveis e avaliando quantidade de clusters global...")
X_escalado = StandardScaler().fit_transform(df_global[COLUNAS_MODELO])
df_z = pd.DataFrame(X_escalado, columns=[f"{col}_z" for col in COLUNAS_MODELO], index=df_global.index)
df_global = pd.concat([df_global, df_z], axis=1)

print("\n--- VALIDAÇÃO GLOBAL: ÍNDICE DE SILHUETA ---")
resultados_silhueta = []
limite_k = min(14, len(df_global) - 1)

for k in range(2, limite_k + 1):
    km_teste = KMeans(n_clusters=k, random_state=42, n_init=10)
    score = silhouette_score(X_escalado, km_teste.fit_predict(X_escalado))
    resultados_silhueta.append({"k": k, "silhueta": score})
    print(f"k={k}: {score:.3f}")
print("--------------------------------------------")

salvar_csv(pd.DataFrame(resultados_silhueta), "04_silhueta_global.csv", index=False)

print(f"\n7. Aplicando K-Means global com k={K_GLOBAL}...")
df_global["Cluster_Original"] = KMeans(n_clusters=K_GLOBAL, random_state=42, n_init=10).fit_predict(X_escalado)
ordem_erro = df_global.groupby("Cluster_Original")["taxa_erro"].mean().sort_values()
mapa_ordenacao = {cluster_antigo: novo_id for novo_id, cluster_antigo in enumerate(ordem_erro.index)}

df_global["Cluster"] = df_global["Cluster_Original"].map(mapa_ordenacao)
df_global["perfil_global"] = df_global["Cluster"].map({0: "Consolidado", 1: "Intermediário", 2: "Crítico"})
print("-> Perfis globais mapeados com sucesso.")

print("\n8. Calculando score auxiliar de prioridade local...")
df_global["score_prioridade_local"] = (
    PESOS_SCORE["taxa_erro_z"] * df_global["taxa_erro_z"] +
    PESOS_SCORE["sec_first_response_z"] * df_global["sec_first_response_z"] +
    PESOS_SCORE["attempt_count_z"] * df_global["attempt_count_z"] +
    PESOS_SCORE["hint_count_z"] * df_global["hint_count_z"]
)
print("-> Score auxiliar calculado.\n   Observação: o score não define o cluster; ele apenas ordena prioridades dentro do módulo.")

print("\n9. Gerando hipóteses comportamentais sobre padrões de erro...")
df_global = adicionar_hipoteses_comportamentais(df_global)
df_resumo_hipoteses = df_global.groupby(["perfil_global", "hipotese_padrao_erro", "forca_indicio_hipotese"]).size().reset_index(name="qtd_topicos").sort_values(["perfil_global", "qtd_topicos"], ascending=[True, False])

print("\nResumo de hipóteses por perfil global:")
print(df_resumo_hipoteses.to_string(index=False))
salvar_csv(df_resumo_hipoteses, "05_resumo_hipoteses_comportamentais_por_perfil.csv", index=False)

print("\n" + "=" * 100)
print("RESUMO ESTATÍSTICO GLOBAL DOS PERFIS")
print("=" * 100)
resumo_perfis = df_global.groupby("perfil_global").agg(
    Qtd_Topicos=("taxa_erro", "count"), Taxa_Erro_Media=("taxa_erro", "mean"),
    Tempo_Medio_Seg=("sec_first_response", "mean"), Tentativas_Media=("attempt_count", "mean"),
    Dicas_Media=("hint_count", "mean"), Score_Medio=("score_prioridade_local", "mean")
).round(3)

ordem_presente = [p for p in ["Consolidado", "Intermediário", "Crítico"] if p in resumo_perfis.index]
resumo_perfis = resumo_perfis.reindex(ordem_presente)
print(resumo_perfis.to_string())
salvar_csv(resumo_perfis.reset_index(), "06_resumo_perfis_globais.csv", index=False)
salvar_csv(df_global.reset_index().rename(columns={"skill": "topico"}), "07_base_agregada_com_clusters_score_hipoteses.csv", index=False)

if RODAR_DIAGNOSTICO_CLUSTER_LOCAL:
    df_diagnostico_local = avaliar_clusterizacao_local_por_modulo(df_global, modulos_pedagogicos, COLUNAS_MODELO, k_max=5)
    salvar_csv(df_diagnostico_local, "08_diagnostico_clusterizacao_local_por_modulo.csv", index=False)

modulos_para_analisar = modulos_pedagogicos if ANALISAR_TODOS_MODULOS else {MODULO_ATUAL: modulos_pedagogicos[MODULO_ATUAL]}
resultados_todos_modulos = {nome: validar_modulo(df_global, nome, skills, TOP_N) for nome, skills in modulos_para_analisar.items()}

df_resumo_comparativo = gerar_resumo_comparativo(resultados_todos_modulos)
salvar_csv(df_resumo_comparativo, "09_resumo_comparativo_modulos.csv", index=False)
gerar_csvs_por_modulo(resultados_todos_modulos)

print("\n" + "=" * 100)
print("EXECUÇÃO FINALIZADA")
print("=" * 100)
print(f"Arquivos gerados em: {PASTA_SAIDA}\n")
print("Principais arquivos csv")
print("- 01_resumo_limpeza.csv\n- 03_resumo_cobertura_modulos.csv\n- 04_silhueta_global.csv")
print("- 05_resumo_hipoteses_comportamentais_por_perfil.csv\n- 06_resumo_perfis_globais.csv")
print("- 07_base_agregada_com_clusters_score_hipoteses.csv\n- 08_diagnostico_clusterizacao_local_por_modulo.csv")
print("- 09_resumo_comparativo_modulos.csv\n- 10_relatorio_detalhado_todos_modulos.csv")
print("- 11_topicos_recomendados_metodo_proposto.csv\n- 12_topicos_baseline_taxa_erro.csv")

1. Conectando ao Google Drive...
Mounted at /content/drive
-> Pasta de saída: /content/drive/MyDrive/resultados_kdd_modulos_sem_afeto

2. Verificando colunas obrigatórias...
-> Todas as colunas obrigatórias foram encontradas.

3. Carregando base de dados geral...
-> Linhas brutas carregadas: 6,123,270

4. Executando pré-processamento...

RESUMO DA LIMPEZA
                                               etapa  linhas  removidas_desde_etapa_anterior  percentual_da_base_original
                                      Base carregada 6123270                             0.0                       100.00
                   Remoção de skill ausente ou vazia 2630080                       3493190.0                        42.95
Remoção de valores ausentes nas variáveis principais 2630080                             0.0                        42.95
         Remoção de tempo/tentativas/dicas inválidos 2630079                             1.0                        42.95
-> Arquivo salvo: /content/drive